In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install mlflow xgboost -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 60.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 9.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

In [3]:
np.random.seed(42)

n_samples = 10000

temperature = np.random.normal(75,15,n_samples)
vibration = np.random.normal(0.5,0.2,n_samples)
pressure = np.random.normal(100,20,n_samples)
rpm = np.random.normal(1500,200,n_samples)
age_days = np.random.randint(0,365,n_samples)

failure_score = (
    (temperature > 90)*0.3 +
    (vibration > 0.8)*0.3 +
    (pressure > 130)*0.2 +
    (age_days > 300)*0.2
)

failure_prob = failure_score + np.random.normal(0,0.1,n_samples)

failure = (failure_prob > 0.35).astype(int)

data = pd.DataFrame({
    'temperature': temperature,
    'vibration': vibration,
    'pressure': pressure,
    'rpm': rpm,
    'age_days': age_days,
    'failure': failure
})

print(data.shape)


(10000, 6)


In [4]:
X = data.drop('failure', axis=1)
y = data['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(8000, 5)
(2000, 5)


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling Complete")

Scaling Complete


In [6]:
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:,1]

lr_auc = roc_auc_score(y_test, lr_prob)

print("Logistic Regression ROC AUC:", lr_auc)

Logistic Regression ROC AUC: 0.8460205629529749


In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:,1]

rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC AUC:", rf_auc)

Random Forest ROC AUC: 0.9421900106747569


In [8]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train)

xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:,1]

xgb_auc = roc_auc_score(y_test, xgb_prob)

print("XGBoost ROC AUC:", xgb_auc)

XGBoost ROC AUC: 0.9413405247485813


In [9]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "ROC_AUC": [
        lr_auc,
        rf_auc,
        xgb_auc
    ]
})

results = results.sort_values(
    "ROC_AUC",
    ascending=False
)

results

,Model,ROC_AUC
1,Random Forest,0.942190
2,XGBoost,0.941341
0,Logistic Regression,0.846021


In [10]:
model_name = "PredictiveMaintenance"

registry = {
    model_name: {
        "version": 1,
        "model": xgb_model,
        "stage": "None"
    }
}

print("Registered:")
print(model_name)
print("Version:", registry[model_name]["version"])

Registered:
PredictiveMaintenance
Version: 1


In [11]:
registry[model_name]["description"] = """
XGBoost model for predictive maintenance.

Features:
temperature
vibration
pressure
rpm
age_days
"""

registry[model_name]["tags"] = {
    "validation_status": "passed",
    "team": "data-science",
    "framework": "xgboost"
}

print("Documentation Added")

Documentation Added


In [12]:
registry[model_name]["stage"] = "Staging"

print("Model moved to Staging")

Model moved to Staging


In [13]:
test_data = pd.DataFrame({
    'temperature':[95],
    'vibration':[0.9],
    'pressure':[135],
    'rpm':[1500],
    'age_days':[320]
})

test_scaled = scaler.transform(test_data)

prediction = xgb_model.predict(test_scaled)

print("Prediction:", prediction[0])

if prediction[0] == 1:
    print("FAILURE LIKELY")
else:
    print("NO FAILURE")

Prediction: 1
FAILURE LIKELY


In [14]:
registry[model_name]["stage"] = "Production"

print("Version 1 is now in Production")

Version 1 is now in Production


In [16]:
def predict_equipment_failure(
    temperature,
    vibration,
    pressure,
    rpm,
    age_days
):

    input_data = pd.DataFrame([{
        'temperature': temperature,
        'vibration': vibration,
        'pressure': pressure,
        'rpm': rpm,
        'age_days': age_days
    }])

    input_scaled = scaler.transform(input_data)

    prediction = xgb_model.predict(input_scaled)[0]

    return {
        "will_fail": bool(prediction),
        "recommendation":
        "Schedule maintenance"
        if prediction
        else
        "Normal operation"
    }

In [17]:
scenarios = [

    {
        "name":"Normal",
        "temp":70,
        "vib":0.4,
        "press":95,
        "rpm":1500,
        "age":100
    },

    {
        "name":"High Risk",
        "temp":95,
        "vib":0.9,
        "press":135,
        "rpm":1500,
        "age":320
    },

    {
        "name":"Medium",
        "temp":85,
        "vib":0.6,
        "press":110,
        "rpm":1500,
        "age":200
    }
]
for s in scenarios:

    result = predict_equipment_failure(
        s["temp"],
        s["vib"],
        s["press"],
        s["rpm"],
        s["age"]
    )

    print(
        s["name"],
        "→",
        result["recommendation"]
    )

Normal → Normal operation
High Risk → Schedule maintenance
Medium → Normal operation


In [18]:
registry["PredictiveMaintenance_v2"] = {
    "version": 2,
    "model": rf_model,
    "stage": "None"
}

print("Version 2 Registered")

Version 2 Registered


In [19]:
production_version = 1

print("Rolling Back...")

print(
    "Production now uses Version",
    production_version
)

Rolling Back...
Production now uses Version 1
